# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Học viên:** Lương Thị Linh
**Mã sinh viên:** 2A202601015
**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 60` cho run nhanh (có thể tăng tối đa 400)

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `OPENAI_API_KEY`, `JUDGE_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [2]:
#@title 1.1 — Install dependencies khi còn thiếu

import importlib.util
import subprocess
import sys

# import_name -> pip_package
required_modules = {
    "neo4j": "neo4j",
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "openai": "openai",
    "datasets": "datasets",
    "dotenv": "python-dotenv",
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_modules.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages
    ])

    print("✅ Dependencies installed.")
else:
    print("✅ Dependencies are already installed.")

Installing missing packages: ['neo4j', 'faiss-cpu', 'groq']
✅ Dependencies installed.


In [4]:
#@title 1.2 — Imports & config

import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss
from dotenv import load_dotenv
from graphrag_bonus import (
    build_community_reports, find_near_duplicate_pairs,
    select_community_reports, traversal_edge_limit,
)


# =========================
# Paths
# =========================

# Use the active repository/workspace in both local Jupyter and Colab.
PROJECT_ROOT = Path.cwd()

load_dotenv(
    PROJECT_ROOT / ".env",
    override=False
)

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = PROJECT_ROOT / "reports"

for directory in (
    DATA_DIR,
    OUTPUT_DIR,
    REPORT_DIR
):
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# =========================
# Reproducibility
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

pd.set_option(
    "display.max_colwidth",
    120
)


# =========================
# Secrets
# =========================

def get_secret(name, default=None):
    try:
        from google.colab import userdata

        value = userdata.get(name)

        if value is not None and value != "":
            return value

    except Exception:
        pass

    return os.environ.get(
        name,
        default
    )


# Neo4j
NEO4J_URI = get_secret(
    "NEO4J_URI",
    ""
)

NEO4J_USER = get_secret(
    "NEO4J_USER",
    "neo4j"
)

NEO4J_PASSWORD = get_secret(
    "NEO4J_PASSWORD",
    ""
)

NEO4J_DATABASE = get_secret(
    "NEO4J_DATABASE",
    ""
)


# OpenAI only
OPENAI_API_KEY = get_secret(
    "OPENAI_API_KEY",
    ""
)

JUDGE_MODEL = get_secret(
    "JUDGE_MODEL",
    "gpt-4o-mini"
)


# Hugging Face
HF_TOKEN = get_secret(
    "HF_TOKEN",
    ""
)


# =========================
# Dataset config
# =========================

DATA_PATH = (
    DATA_DIR /
    "hackernoon_subset.csv"
)

LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000

EXTRACTION_MAX_CHUNKS = 60  # Có thể tăng đến 400 khi quota/thời gian cho phép

CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


# =========================
# Validation
# =========================

def validate_config():

    required = {
        "NEO4J_URI":
            NEO4J_URI,

        "NEO4J_PASSWORD":
            NEO4J_PASSWORD,

        "OPENAI_API_KEY":
            OPENAI_API_KEY,

        # HF token is required only when the local CSV has to be streamed.
        **({"HF_TOKEN": HF_TOKEN} if not DATA_PATH.exists() else {}),
    }

    missing = [
        name
        for name, value
        in required.items()
        if not value
    ]

    if missing:
        raise RuntimeError(
            "Thiếu cấu hình: "
            + ", ".join(missing)
        )

    print(
        "✅ Cấu hình hợp lệ."
    )

    print(
        "✅ LLM/Judge: OpenAI"
    )

    print(
        f"✅ Judge model: {JUDGE_MODEL}"
    )

    print(
        "✅ Không có secret nào được in ra."
    )


validate_config()

✅ Cấu hình hợp lệ.
✅ LLM/Judge: OpenAI
✅ Judge model: gpt-4o-mini
✅ Không có secret nào được in ra.


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [5]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = str(DATA_PATH)

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = LAB_MAX_ARTICLES
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = False

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Đang ghi dữ liệu vào: /content/data/hackernoon_subset.csv


Đang tải (row):   0%|          | 0/1500 [00:00<?, ?row/s]


[DỪNG] Đã đạt giới hạn số dòng: 1,500 dòng (Dung lượng: 0.89 MB)
✅ Hoàn thành: /content/data/hackernoon_subset.csv
   Rows: 1,500
   Size: 0.89 MB


In [6]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.
✅ Schema ready.


In [7]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    value = unicodedata.normalize("NFKC", str(x or ""))
    return re.sub(r"\s+", " ", value).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "articleBody", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at", "createdAt", "datePublished"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid", "objectID"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)

    # Bonus A: LSH produces reviewable candidates without O(N^2) comparisons.
    near_pairs = find_near_duplicate_pairs(
        (df['title'] + '\n' + df['text']).tolist(), threshold=0.92
    )
    near_audit = pd.DataFrame(near_pairs)
    if not near_audit.empty:
        near_audit['decision'] = 'MERGE_NEAR_DUP'
        near_audit['left_article_id'] = near_audit.left_index.map(df.article_id)
        near_audit['right_article_id'] = near_audit.right_index.map(df.article_id)
        # Keep the earliest row in each detected pair; the audit preserves every decision.
        df = df.drop(index=sorted(set(near_audit.right_index))).reset_index(drop=True)
    else:
        near_audit = pd.DataFrame(columns=['left_index','right_index','similarity','hamming_distance','decision','left_article_id','right_article_id'])
    print(f'Near dedup (SimHash-LSH): {len(near_audit):,} reviewed pairs; {len(df):,} retained articles')
    return df, near_audit

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df, near_dedup_audit_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
assert not chunks_df.empty, 'Không tạo được chunk nào từ dữ liệu.'
display(chunks_df.head())

Exact dedup: 785 -> 765


Chunking:   0%|          | 0/765 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,ec98609611765e97440d::c0000,ec98609611765e97440d,Adobe student receives national Information and Technology award,2023-05-02,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...
2,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
3,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
4,e1dfbe88dae03136f847::c0000,e1dfbe88dae03136f847,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023,2023-05-15,The conference will bring together growth oriented publicly traded clean energy and technology companies ... up to 4...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [8]:
#@title 1.6 — OpenAI wrapper có retry + JSON parsing

from openai import OpenAI

openai_client = (
    OpenAI(api_key=OPENAI_API_KEY)
    if OPENAI_API_KEY
    else None
)


def parse_json_object(text):
    text = str(text).strip()

    # bỏ markdown code fence nếu model trả về ```json ... ```
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )
    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    a = text.find("{")
    b = text.rfind("}")

    if a < 0 or b <= a:
        raise ValueError(
            "No JSON object found."
        )

    return json.loads(
        text[a:b+1]
    )


def openai_chat(
    messages,
    model=None,
    json_mode=False,
    max_retries=4
):
    if openai_client is None:
        raise RuntimeError(
            "Thiếu OPENAI_API_KEY."
        )

    model = model or JUDGE_MODEL or "gpt-4o-mini"

    last = None

    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }

            # ép model trả JSON object
            if json_mode:
                kwargs["response_format"] = {
                    "type": "json_object"
                }

            resp = (
                openai_client
                .chat
                .completions
                .create(**kwargs)
            )

            usage = {}

            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens":
                        getattr(
                            resp.usage,
                            "prompt_tokens",
                            None
                        ),

                    "completion_tokens":
                        getattr(
                            resp.usage,
                            "completion_tokens",
                            None
                        ),

                    "total_tokens":
                        getattr(
                            resp.usage,
                            "total_tokens",
                            None
                        ),
                }

            text = (
                resp
                .choices[0]
                .message
                .content
            )

            return text, usage

        except Exception as e:
            last = e

            if attempt == max_retries - 1:
                break

            time.sleep(
                min(
                    20,
                    2**attempt + random.random()
                )
            )

    raise RuntimeError(
        f"OpenAI call failed: {last}"
    )


def openai_json(
    system,
    user,
    model=None
):
    text, usage = openai_chat(
        [
            {
                "role": "system",
                "content": system
            },
            {
                "role": "user",
                "content": user
            }
        ],
        model=model,
        json_mode=True,
    )

    return (
        parse_json_object(text),
        usage
    )

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [9]:
#@title 1.7 — Coreference resolution theo batch

COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()


def resolve_coref_batch(batch_df):

    payload = [
        {
            "chunk_id": r.chunk_id,
            "text": r.text
        }
        for r in batch_df.itertuples(index=False)
    ]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    # OpenAI wrapper with retry and JSON parsing
    obj, usage = openai_json(
        COREF_SYSTEM,
        prompt,
        model=JUDGE_MODEL
    )

    by_id = {
        x.get("chunk_id"): x
        for x in obj.get("items", [])
    }

    rows = []

    for r in batch_df.itertuples(index=False):

        item = by_id.get(
            r.chunk_id,
            {}
        )

        rows.append({
            "chunk_id":
                r.chunk_id,

            "resolved_text":
                norm_space(
                    item.get("resolved_text")
                    or r.text
                ),

            "unresolved_mentions":
                item.get(
                    "unresolved_mentions",
                    []
                ),
        })

    return (
        pd.DataFrame(rows),
        usage
    )


def run_coref(
    chunks_subset,
    batch_size=5
):

    out = []

    for start in tqdm(
        range(
            0,
            len(chunks_subset),
            batch_size
        ),
        desc="Coref"
    ):

        batch = chunks_subset.iloc[
            start:start + batch_size
        ]

        try:

            df, _ = resolve_coref_batch(
                batch
            )

        except Exception as e:

            print(
                f"⚠️ Coref batch failed: "
                f"{type(e).__name__}"
            )

            df = pd.DataFrame({
                "chunk_id":
                    batch["chunk_id"].tolist(),

                "resolved_text":
                    batch["text"].tolist(),

                "unresolved_mentions":
                    [
                        ["COREF_BATCH_FAILED"]
                        for _ in range(len(batch))
                    ],
            })

        out.append(df)

    return pd.concat(
        out,
        ignore_index=True
    )


# =========================
# Run coreference
# =========================

extraction_source = (
    chunks_df
    .head(EXTRACTION_MAX_CHUNKS)
    .copy()
)

coref_cache = (
    DATA_DIR /
    "coref_results.jsonl"
)


# =========================
# Cache
# =========================

if coref_cache.exists():

    cached_coref = pd.read_json(
        coref_cache,
        lines=True
    )

    wanted = set(
        extraction_source.chunk_id
    )

    coref_df = cached_coref[
        cached_coref.chunk_id.isin(wanted)
    ].copy()

    # cache thiếu chunk -> chạy lại
    if set(coref_df.chunk_id) != wanted:

        coref_df = run_coref(
            extraction_source
        )

else:

    coref_df = run_coref(
        extraction_source
    )


# =========================
# Save
# =========================

coref_df.to_json(
    coref_cache,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================
# Merge
# =========================

extraction_source = (
    extraction_source
    .merge(
        coref_df,
        on="chunk_id",
        how="left"
    )
)


# =========================
# Summary
# =========================

unresolved_count = sum(
    len(x)
    for x in coref_df.unresolved_mentions
)

print(
    f"✅ Coreference: "
    f"{len(coref_df):,} chunks; "
    f"{unresolved_count:,} unresolved mentions."
)

display(
    coref_df.head()
)

Coref:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Coreference: 60 chunks; 4 unresolved mentions.


,chunk_id,resolved_text,unresolved_mentions
0,1a05beb7aa3071be6fd7::c0000,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...,[]
1,ec98609611765e97440d::c0000,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...,[]
2,8e922bc62b578e73e815::c0000,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...,[]
3,4bd7afdba71243b0dbcd::c0000,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...,[]
4,e1dfbe88dae03136f847::c0000,The conference will bring together growth oriented publicly traded clean energy and technology companies ... up to 4...,[]


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [10]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return openai_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "unknown",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": max(0.0, min(1.0, float(x.get("confidence") or 0.0))),
                })

    triples_df = pd.DataFrame(triples)
    if not triples_df.empty:
        triples_df = triples_df[triples_df.evidence.str.len() > 0].reset_index(drop=True)
    return triples_df, pd.DataFrame(errors)

triple_cache = DATA_DIR / 'raw_triples.csv'
error_cache = DATA_DIR / 'extraction_errors.csv'
if triple_cache.exists():
    raw_triples_df = pd.read_csv(triple_cache)
    extraction_errors_df = pd.read_csv(error_cache) if error_cache.exists() and error_cache.stat().st_size > 0 else pd.DataFrame()
else:
    raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
    raw_triples_df.to_csv(triple_cache, index=False)
    if not extraction_errors_df.empty:
        extraction_errors_df.to_csv(error_cache, index=False)
assert not raw_triples_df.empty, 'Không trích xuất được triple hợp lệ.'
print(f'Triples: {len(raw_triples_df):,}; failed batches: {len(extraction_errors_df):,}')
display(raw_triples_df.head())

NER+RE:   0%|          | 0/15 [00:00<?, ?it/s]

Triples: 26; failed batches: 0


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Sineng Electric,Company,PARTNERED_WITH,onsemi,Company,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC silic,0.9
1,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,GreenPages acquired Toronto-based Zanaris,0.9
2,Aeris Communications,Company,PARTNERED_WITH,Ericsson,Company,4f1346392056a403277d::c0000,2022-12-07,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry.,1.0
3,Walt Disney Co.,Company,LEADS,Bob Iger,Person,5d0b9cbc04b00a5fe84a::c0000,2023-08-10,Walt Disney Co. CEO Bob Iger vowed to make Walt Disney Co.'s streaming services profitable.,1.0
4,Sojern,Company,ACQUIRED,VenueLytics,Company,ea275b79c4b3b46d5941::c0000,2023-07-11,Sojern has acquired VenueLytics a platform that provides guest management and communications software for independen...,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [11]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
    "open ai": "OpenAI",
    "huggingface": "Hugging Face",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b, entity_type):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    ta, tb = na.split(), nb.split()
    if not ta or not tb:
        return False
    if entity_type == 'Person':
        same_last_name = ta[-1] == tb[-1]
        same_first_name = ta[0] == tb[0] or ta[0][:1] == tb[0][:1] and (len(ta[0]) == 1 or len(tb[0]) == 1)
        return same_last_name and same_first_name
    if entity_type == 'Company':
        return SequenceMatcher(None, na, nb).ratio() >= 0.82
    return SequenceMatcher(None, na, nb).ratio() >= 0.90

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=10, audit_floor=-1.0):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if t == 'Company' and norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < audit_floor:
                    continue
                above_threshold = float(score) >= threshold
                ok = above_threshold and merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else ("REJECT_GUARD" if above_threshold else "REJECT_THRESHOLD")
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
entity_resolution_audit_df['audit_scope'] = 'DATA'
# Controlled policy probe: high vector similarity must not bypass the lexical guard.
probe_left, probe_right = 'Samsung Electronics', 'Samsung Semiconductor Electronics'
probe_vectors = get_embedder().encode([probe_left, probe_right], normalize_embeddings=True)
probe_similarity = float(probe_vectors[0] @ probe_vectors[1])
probe_decision = ('MERGE_VECTOR' if probe_similarity >= 0.90 and merge_guard(probe_left, probe_right, 'Company')
                  else 'REJECT_GUARD' if probe_similarity >= 0.90 else 'REJECT_THRESHOLD')
policy_probe_df = pd.DataFrame([{'type':'Company', 'left':probe_left, 'right':probe_right,
                                 'similarity':probe_similarity, 'decision':probe_decision,
                                 'audit_scope':'CONTROLLED_POLICY_TEST'}])
entity_resolution_audit_df = pd.concat([entity_resolution_audit_df, policy_probe_df], ignore_index=True)
assert probe_similarity > 0.85 and probe_decision == 'REJECT_GUARD'
triples_df = canonicalize_triples(raw_triples_df, entity_map)
assert len(entity_resolution_audit_df) >= 10, 'Entity-resolution audit cần ít nhất 10 dòng.'
print(entity_resolution_audit_df.decision.value_counts())
display(entity_resolution_audit_df.head(20))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

decision
REJECT_THRESHOLD    191
REJECT_GUARD          1
Name: count, dtype: int64


,type,left,right,similarity,decision,audit_scope
0,Company,Sineng Electric,Samsung Electronics Co. Ltd.,0.343637,REJECT_THRESHOLD,DATA
1,Company,Sineng Electric,Crexendo,0.284176,REJECT_THRESHOLD,DATA
2,Company,Sineng Electric,telecom operators,0.281158,REJECT_THRESHOLD,DATA
3,Company,Sineng Electric,Intelligent Technical Solutions,0.276068,REJECT_THRESHOLD,DATA
4,Company,Sineng Electric,ACP IT Solutions GmbH Dresden,0.269209,REJECT_THRESHOLD,DATA
5,Company,Sineng Electric,Iridium Communications Inc.,0.265260,REJECT_THRESHOLD,DATA
6,Company,Sineng Electric,Citi,0.252985,REJECT_THRESHOLD,DATA
7,Company,Sineng Electric,onsemi,0.243364,REJECT_THRESHOLD,DATA
8,Company,Sineng Electric,dynaCERT,0.240035,REJECT_THRESHOLD,DATA
9,Company,onsemi,Citi,0.350529,REJECT_THRESHOLD,DATA


In [12]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print(f'✅ Inserted/updated {len(nodes_df):,} nodes and {len(triples_df):,} provenance edges.')

✅ Inserted/updated 46 nodes and 26 provenance edges.


In [13]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR trim(toString(r.source_chunk_id)) = ''
       OR r.published_date IS NULL OR trim(toString(r.published_date)) = ''
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 46, 'edges': 26, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,b4fe88d52bfdb340ad30a63c,video streaming services,Technology,3
1,b79293f3033084556da650c0,Information Services Group,Company,2
2,23d7ceb58c062d83135817ba,Intelligent Technical Solutions,Company,2
3,f5a771e89e84463402aaeff5,Citi Token Services,Technology,2
4,e6c3db5c28c9a15bfafd2e04,Apple,Company,2
5,1c217d9e4adb70fc731cb387,Walt Disney Co.,Company,1
6,13560ce11a7425ac7d706b75,BrightWire Networks,Company,1
7,0430a89df77e6234ef375307,GreenPages,Company,1
8,5c0dfda16cbfa608d8cfc87b,IDC,Company,1
9,431cca765017665a7961a6ac,Aretum,Company,1


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [14]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    if flat_index is None or flat_store is None:
        raise RuntimeError('Flat index chưa được khởi tạo.')
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Flat vectors: 765


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [17]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = openai_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [18]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY CASE WHEN r.published_date = 'unknown' THEN '' ELSE coalesce(r.published_date,'') END DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    def date_key(edge):
        value = edge.get("published_date") or ""
        return "" if value == "unknown" else value
    edges = sorted(edges, key=date_key, reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = traversal_edge_limit(
            degree, edge_limit, SUPER_NODE_DEGREE, SUPER_NODE_EDGE_CAP
        )
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [21]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = openai_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=JUDGE_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [22]:
#@title 4.1 — Golden Dataset có đáp án truy vết từ dữ liệu
GOLDEN_PATH = DATA_DIR / "golden_dataset.csv"
RELATION_TEXT = {
    'ACQUIRED':'acquire', 'DEVELOPED':'develop', 'INVESTED_IN':'invest in',
    'FOUNDED':'found', 'WORKED_AT':'work at', 'PARTNERED_WITH':'partner with',
    'USES':'use', 'LEADS':'lead',
}

def edge_statement(r):
    return f"{r.source_name} -{r.relation}-> {r.target_name} ({r.published_date})"

def build_golden_dataset(triples):
    df = (triples.sort_values(['confidence','published_date'], ascending=[False,False])
          .drop_duplicates(['source_id','relation','target_id','source_chunk_id'])
          .reset_index(drop=True))
    if len(df) < 5:
        raise RuntimeError('Cần ít nhất 5 triple để tạo Golden Dataset.')

    rows = []
    factoids = df.drop_duplicates(['source_id','relation']).head(2)
    if len(factoids) < 2:
        raise RuntimeError('Không đủ hai fact riêng biệt để tạo Golden Dataset.')
    for r in factoids.itertuples(index=False):
        related = df[(df.source_id == r.source_id) & (df.relation == r.relation)].head(3)
        statements = [edge_statement(x) for x in related.itertuples(index=False)]
        evidence = [f"[{x.source_chunk_id}] {x.evidence}" for x in related.itertuples(index=False)]
        target_types = sorted(set(related.target_type.str.lower()))
        target_label = target_types[0] if len(target_types) == 1 else 'entities'
        rows.append({
            'group':'factoid',
            'question':f"According to the indexed news, which {target_label} did {r.source_name} {RELATION_TEXT[r.relation]}?",
            'reference_answer':'; '.join(statements),
            'reference_evidence':' | '.join(evidence),
        })

    chains = []
    edge_rows = list(df.head(200).itertuples(index=False))
    for i, left in enumerate(edge_rows):
        left_nodes = {left.source_id:left.source_name, left.target_id:left.target_name}
        for right in edge_rows[i+1:]:
            right_nodes = {right.source_id:right.source_name, right.target_id:right.target_name}
            shared = set(left_nodes) & set(right_nodes)
            if not shared:
                continue
            middle_id = next(iter(shared))
            left_other = next((x for x in left_nodes if x != middle_id), None)
            right_other = next((x for x in right_nodes if x != middle_id), None)
            if not left_other or not right_other or left_other == right_other:
                continue
            chains.append((left, right, left_nodes[left_other], left_nodes[middle_id], right_nodes[right_other]))
            if len(chains) == 2:
                break
        if len(chains) == 2:
            break
    if len(chains) < 2:
        raise RuntimeError('Không đủ hai chuỗi quan hệ hai-hop; hãy tăng EXTRACTION_MAX_CHUNKS.')
    for left, right, start_name, middle_name, end_name in chains:
        rows.append({
            'group':'multi-hop',
            'question':f"Using two linked facts, how is {start_name} connected to {end_name} through {middle_name}?",
            'reference_answer':f"{edge_statement(left)}; {edge_statement(right)}.",
            'reference_evidence':f"[{left.source_chunk_id}] {left.evidence} | [{right.source_chunk_id}] {right.evidence}",
        })

    entity_counts = Counter()
    for r in df.itertuples(index=False):
        entity_counts[r.source_name] += 1
        entity_counts[r.target_name] += 1
    cross_doc = None
    for entity, _ in entity_counts.most_common():
        events = df[(df.source_name == entity) | (df.target_name == entity)].drop_duplicates('source_chunk_id').head(3)
        if events.source_chunk_id.nunique() >= 2:
            cross_doc = (entity, events)
            break
    if cross_doc is None:
        events = df.drop_duplicates('source_chunk_id').head(2)
        if len(events) < 2:
            raise RuntimeError('Không đủ hai tài liệu để tạo câu hỏi cross-document.')
        event_names = ' and '.join(dict.fromkeys(events.source_name.astype(str)))
        question = f"Across two different news chunks, what relationships were reported for {event_names}?"
    else:
        entity, events = cross_doc
        question = f"What relationships involving {entity} are supported by at least two different news chunks?"
    statements = [edge_statement(r) for r in events.itertuples(index=False)]
    evidence = [f"[{r.source_chunk_id}] {r.evidence}" for r in events.itertuples(index=False)]
    rows.append({
        'group':'cross-doc',
        'question':question,
        'reference_answer':' '.join(statements),
        'reference_evidence':' | '.join(evidence),
    })

    out = pd.DataFrame(rows)
    out.insert(0, 'id', [f'G{i:02d}' for i in range(1, len(out)+1)])
    return out

rebuild_golden = os.getenv('REBUILD_GOLDEN', 'true').lower() == 'true'
golden_df = build_golden_dataset(triples_df) if rebuild_golden or not GOLDEN_PATH.exists() else pd.read_csv(GOLDEN_PATH)
golden_df.to_csv(GOLDEN_PATH, index=False)
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    required_groups = {'factoid','multi-hop','cross-doc'}
    if not required_groups.issubset(set(df.group)):
        raise ValueError(f"Golden Dataset thiếu nhóm: {required_groups-set(df.group)}")
    if len(df) < 5:
        raise ValueError('Golden Dataset cần ít nhất 5 câu hỏi.')
    print("✅ Golden Dataset valid.")

validate_golden(golden_df, require_answers=True)

,id,group,question,reference_answer,reference_evidence
0,G01,factoid,"According to the indexed news, which technology did Samsung Electronics Co. Ltd. develop?",Samsung Electronics Co. Ltd. -DEVELOPED-> analog and logic semiconductor technologies (2023-10-05),[8c5930949a3d9f3c3a38::c0000] Samsung Electronics Co. Ltd. ... unveiled its latest innovations in analog and logic s...
1,G02,factoid,"According to the indexed news, which technology did Citi develop?",Citi -DEVELOPED-> Citi Token Services (2023-09-26),[3ac1a0d4fdc98952e374::c0000] Citi has launched a token service using blockchain technology to offer digital asset s...
2,G03,multi-hop,"Using two linked facts, how is Citi connected to blockchain technology through Citi Token Services?",Citi -DEVELOPED-> Citi Token Services (2023-09-26); Citi Token Services -USES-> blockchain technology (2023-09-26).,[3ac1a0d4fdc98952e374::c0000] Citi has launched a token service using blockchain technology to offer digital asset s...
3,G04,multi-hop,"Using two linked facts, how is iPhone 15 connected to video streaming services through Apple?",Apple -DEVELOPED-> iPhone 15 (2023-09-19); Apple -PARTNERED_WITH-> video streaming services (2023-06-01).,[8b2a4221bdca74cf28ac::c0000] Apple announced the launch of the iPhone 15 | [54d02d03b666fe038c85::c0000] Apple deci...
4,G05,cross-doc,What relationships involving Apple are supported by at least two different news chunks?,Apple -DEVELOPED-> iPhone 15 (2023-09-19) Apple -PARTNERED_WITH-> video streaming services (2023-06-01),[8b2a4221bdca74cf28ac::c0000] Apple announced the launch of the iPhone 15 | [54d02d03b666fe038c85::c0000] Apple deci...


✅ Golden Dataset valid.


In [23]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")
    if not OPENAI_API_KEY:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")
    return openai_json(system, user, model=JUDGE_MODEL)[0]

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [24]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = OUTPUT_DIR / "graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,"According to the indexed news, which technology did Samsung Electronics Co. Ltd. develop?",Samsung Electronics Co. Ltd. -DEVELOPED-> analog and logic semiconductor technologies (2023-10-05),"Samsung Electronics Co. Ltd. developed advanced semiconductor technology, specifically innovations in analog and log...",Samsung Electronics Co. Ltd. developed analog and logic semiconductor technologies [chunk_id=8c5930949a3d9f3c3a38::c...,5,5,5,5,5,5,1.170064,1.064266,542,473,"The candidate accurately identifies that Samsung Electronics Co. Ltd. developed advanced semiconductor technology, s...",The candidate accurately states that Samsung Electronics Co. Ltd. developed analog and logic semiconductor technolog...,0
1,G02,factoid,"According to the indexed news, which technology did Citi develop?",Citi -DEVELOPED-> Citi Token Services (2023-09-26),"Citi developed a token service called Citi Token Services, which utilizes blockchain technology to offer digital ass...","Citi developed the Citi Token Services, which utilizes blockchain technology to offer digital asset solutions for it...",5,5,5,5,5,5,1.534132,1.052958,547,525,"The candidate accurately identifies the technology developed by Citi as Citi Token Services, which is consistent wit...",The candidate accurately identifies that Citi developed the Citi Token Services and provides additional context abou...,0
2,G03,multi-hop,"Using two linked facts, how is Citi connected to blockchain technology through Citi Token Services?",Citi -DEVELOPED-> Citi Token Services (2023-09-26); Citi Token Services -USES-> blockchain technology (2023-09-26).,"Citi is connected to blockchain technology through its launch of Citi Token Services, which utilizes blockchain to c...","Citi is connected to blockchain technology through its development of Citi Token Services, which utilizes blockchain...",4,4,5,5,4,4,1.148925,1.290962,582,579,The candidate accurately describes the connection between Citi and blockchain technology through Citi Token Services...,The candidate provides a clear connection between Citi and blockchain technology through the development of Citi Tok...,0
3,G04,multi-hop,"Using two linked facts, how is iPhone 15 connected to video streaming services through Apple?",Apple -DEVELOPED-> iPhone 15 (2023-09-19); Apple -PARTNERED_WITH-> video streaming services (2023-06-01).,The iPhone 15 is connected to video streaming services through Apple as the company is actively involved in the vide...,"The iPhone 15, developed by Apple, is connected to video streaming services through Apple's partnership with these s...",3,4,4,5,3,4,3.000874,1.957747,660,725,The candidate provides a connection between the iPhone 15 and video streaming services through Apple's involvement i...,The candidate provides a clear connection between the iPhone 15 and video streaming services through Apple's partner...,0
4,G05,cross-doc,What relationships involving Apple are supported by at least two different news chunks?,Apple -DEVELOPED-> iPhone 15 (2023-09-19) Apple -PARTNERED_WITH-> video streaming services (2023-06-01),The relationships involving Apple that are supported by at least two different news chunks are:\n\n1. **Apple's Role...,The relationships involving Apple that are supported by at least two different news chunks are:\n\n1. **Apple develo...,2,5,3,5,2,5,2.336082,2.145638,661,738,"The candidate identifies two relationships involving Apple, but they do not align with the reference answer. The fir...","The candidate provides a complete and accurate summary of the relationships involving Apple, citing specific news ch...",0


In [25]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    grouped = list(eval_df.groupby("group")) + [('overall', eval_df)]
    for group, g in grouped:
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_path = OUTPUT_DIR / 'graphrag_eval_results.csv'
summary_path = OUTPUT_DIR / 'graphrag_vs_flatrag_summary.csv'
eval_results_df.to_csv(eval_path, index=False)
comparison_df.to_csv(summary_path, index=False)
print(f'✅ Exported {eval_path} and {summary_path}')

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,2.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
1,cross-doc,Faithfulness,3.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
2,cross-doc,Multi-hop reasoning,2.000,5.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
3,cross-doc,Latency (s),2.336,2.146,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,661.000,738.000,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.352,1.059,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,544.500,499.000,GraphRAG không đắt hơn trong sample này.


✅ Exported /content/outputs/graphrag_eval_results.csv and /content/outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [26]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    # Deterministic boundary checks prove the cap even when this small lab graph
    # happens not to contain a degree > 100 node.
    assert traversal_edge_limit(SUPER_NODE_DEGREE, 1000) == 1000
    assert traversal_edge_limit(SUPER_NODE_DEGREE + 1, 1000) == SUPER_NODE_EDGE_CAP
    limit = traversal_edge_limit(n["degree"], 1000, SUPER_NODE_DEGREE, SUPER_NODE_EDGE_CAP)
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= SUPER_NODE_EDGE_CAP
        print("✅ Super-node cap OK.")
    else:
        print("✅ Boundary policy verified with synthetic degree 101 → 50 edges.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': 'b4fe88d52bfdb340ad30a63c', 'name': 'video streaming services', 'degree': 3} fetched= 3


,type,left,right,similarity,decision,audit_scope
191,Company,Samsung Electronics,Samsung Semiconductor Electronics,0.904212,REJECT_GUARD,CONTROLLED_POLICY_TEST
48,Company,Walt Disney Co.,Disney,0.782894,REJECT_THRESHOLD,DATA
176,Technology,cryptocurrencies,blockchain technology,0.489789,REJECT_THRESHOLD,DATA
33,Company,Aeris Communications,Iridium Communications Inc.,0.475362,REJECT_THRESHOLD,DATA
34,Company,Aeris Communications,telecom operators,0.457666,REJECT_THRESHOLD,DATA
71,Company,Samsung Electronics Co. Ltd.,telecom operators,0.456186,REJECT_THRESHOLD,DATA
72,Company,Samsung Electronics Co. Ltd.,Iridium Communications Inc.,0.438389,REJECT_THRESHOLD,DATA
181,Technology,VIP Business Communications Platform,video streaming services,0.437700,REJECT_THRESHOLD,DATA
115,Company,Iridium Communications Inc.,telecom operators,0.424387,REJECT_THRESHOLD,DATA
104,Company,SIOS Technology Corp,Iridium Communications Inc.,0.408921,REJECT_THRESHOLD,DATA


High-similarity rejected pairs:


,type,left,right,similarity,decision,audit_scope
191,Company,Samsung Electronics,Samsung Semiconductor Electronics,0.904212,REJECT_GUARD,CONTROLLED_POLICY_TEST


## 5.2 — Thuyết minh kỹ thuật — Lương Thị Linh

1. **Coreference sai ở tình huống nào?** Run xử lý 60 chunk và ghi nhận 4 unresolved mentions. Với tham chiếu kiểu *the company* trong đoạn có nhiều công ty nhưng không có antecedent rõ trong cùng chunk, pipeline giữ nguyên thay vì đoán. Cách conservative này giảm recall nhẹ nhưng tránh false edge gán nhầm `ACQUIRED` hoặc `PARTNERED_WITH`. Không có false-resolution cụ thể được lưu trong output hiện có.

2. **Entity threshold bao nhiêu, vì sao?** Dùng cosine threshold `0.90`. Ngưỡng cao giúp ANN chỉ tạo candidate thật gần; sau đó lexical guard theo entity type mới cho phép Union-Find merge, giảm false merge có tính dây chuyền.

3. **Candidate nào similarity cao nhưng không nên merge?** `Samsung Electronics` và `Samsung Semiconductor Electronics` có similarity `0.904212` nhưng bị `REJECT_GUARD`. Chúng chia sẻ thương hiệu nhưng thực thể thứ hai là division/sản phẩm cụ thể; gộp sẽ làm mất phân biệt quan hệ.

4. **Top 3 super-node và degree?** Output Neo4j của run: (1) `video streaming services` — Technology, degree 3; (2) `Information Services Group` — Company, degree 2; (3) `Intelligent Technical Solutions` — Company, degree 2. Dataset lab nhỏ nên chưa có degree >100; notebook kiểm tra thêm boundary 100/101 để chứng minh cap 50.

5. **Vì sao ưu tiên edge mới nhất có thể đúng/sai?** Đúng vì chặn context explosion và ưu tiên tin mới ở super-node. Sai với câu hỏi lịch sử vì edge cũ có thể là evidence quyết định. Mitigation là time filter hoặc nới cap có kiểm soát cho truy vấn temporal.

6. **Flat RAG thắng nhóm nào?** Factoid G01–G02: cả hai đều đạt 5/5 trên quality, nên Flat RAG hợp lý hơn khi cần pipeline/index đơn giản và token rẻ hơn.

7. **GraphRAG thắng nhóm nào?** Cross-doc G05: GraphRAG đạt 5/5 ở comprehensiveness, faithfulness và multi-hop reasoning, so với Flat RAG 2/5, 3/5, 2/5; graph gom được hai edge Apple từ các chunk khác nhau kèm provenance.

8. **Latency/token trade-off?** Trung bình run này: Flat 1.838s, 598.4 tokens; Graph 1.502s, 608.0 tokens. Graph nhanh hơn 0.336s trong graph rất nhỏ nhưng tốn hơn 9.6 tokens/câu; không nên suy rộng latency này khi scale.

9. **AI Coding Agent đề xuất gì mà bạn không dùng, vì sao?** Từ chối pairwise cosine trên toàn bộ bài viết cho near-dedup/entity candidates vì O(N²), dễ OOM ở 350MB. Thay vào đó: ANN cho entity resolution và SimHash-LSH cho near-dedup, cả hai đều có audit trước merge.

10. **Scale 350MB: bottleneck đầu tiên là gì?** LLM extraction throughput/rate-limit là bottleneck trước traversal. Cần queue/async workers có retry, checkpoint theo chunk, bulk `UNWIND`, ANN/HNSW cho candidate search, community partitioning và evidence retention policy.

# 🎁 BONUS

## A — Near-dedup bằng SimHash-LSH (+2)
`standardize_news()` dùng SimHash 64-bit + LSH banding để chỉ so sánh các candidate cùng bucket, không dùng pairwise cosine O(N²). Mỗi merge có audit gồm ID hai bài, Hamming distance và similarity.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. tạo report xác định (deterministic) có provenance cho từng community,
5. router query global trên reports, giữ local traversal cho câu hỏi thực thể cụ thể.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [28]:
#@title Bonus B — Community reports + global-search router
import networkx as nx

COMMUNITY_REPORT_PATH = OUTPUT_DIR / 'community_reports.csv'
GLOBAL_QUERY_HINTS = ('overview', 'trend', 'compare', 'comparison', 'relationships', 'across', 'all ')

def build_communities(limit_edges=20000):
    columns = ['source','target','source_name','relation','target_name','source_chunk_id','published_date','evidence']
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target, a.name AS source_name, type(r) AS relation,
           b.name AS target_name, r.source_chunk_id AS source_chunk_id,
           r.published_date AS published_date, r.evidence AS evidence
    LIMIT $limit
    """, limit=int(limit_edges)), columns=columns)
    if edge_df.empty:
        return edge_df, pd.DataFrame(columns=['id','community_id']), pd.DataFrame(columns=['community_id','edge_count','report'])

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    membership = {r['id']: r['community_id'] for r in rows}
    reports = build_community_reports(edge_df, membership)
    reports.to_csv(COMMUNITY_REPORT_PATH, index=False)
    return edge_df, pd.DataFrame(rows), reports

def is_global_query(question):
    text = norm_space(question).lower()
    return any(hint in text for hint in GLOBAL_QUERY_HINTS)

def retrieve_community_context(question, limit=3):
    reports = globals().get('community_reports_df', pd.DataFrame())
    selected = select_community_reports(question, reports, limit=limit)
    return '\n\n=== COMMUNITY REPORT ===\n'.join(selected.report.tolist()), selected

community_edges_df, community_df, community_reports_df = build_communities()
print(f'✅ Community reports: {len(community_reports_df):,}; exported {COMMUNITY_REPORT_PATH.name}')
display(community_reports_df.head())

In [29]:
#@title Bonus C — Self-correcting GraphRAG with bounded fallback
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = openai_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":"","graph_debug":g2}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing,"graph_debug":g3}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2,
        "graph_debug":g3
    }

def answer_graph_rag(question):
    # Global questions use compact community reports; entity questions use bounded
    # self-correction: hop 2, then hop 3, then vector fallback.
    if is_global_query(question) and not community_reports_df.empty:
        report_context, selected = retrieve_community_context(question)
        vctx, vdocs = retrieve_flat_context(question, k=4)
        context = f"=== COMMUNITY REPORTS ===\n{report_context}\n\n=== VECTOR ===\n{vctx}"
        out = generate_answer(question, context)
        out.update({
            'context': context, 'route': 'community+vector',
            'community_reports': selected, 'vector_docs': vdocs,
            'graph_debug': {'diagnostics': {'supernode_events': []}},
        })
        return out

    adaptive = self_correcting_context(question)
    if adaptive['route'] != 'hop3+vector':
        vctx, vdocs = retrieve_flat_context(question, k=4)
        context = f"=== GRAPH ===\n{adaptive['context']}\n\n=== VECTOR ===\n{vctx}"
    else:
        context, vdocs = adaptive['context'], None
    out = generate_answer(question, context)
    out.update({
        'context': context, 'route': adaptive['route'], 'missing': adaptive['missing'],
        'graph_debug': adaptive['graph_debug'], 'vector_docs': vdocs,
    })
    return out

# One observable smoke check for the bonus route; it does not rerun the full benchmark.
bonus_smoke = answer_graph_rag(golden_df.iloc[0].question)
assert bonus_smoke['route'] in {'hop2', 'hop3', 'hop3+vector', 'community+vector'}
print(f"✅ Bonus retrieval route: {bonus_smoke['route']}")

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [x] Neo4j connected
- [x] Dedup/chunking đã chạy
- [x] Coreference spot-check
- [x] Entity resolution audit
- [x] `UNWIND` bulk insert
- [x] 0 edge thiếu provenance
- [x] Flat RAG chạy
- [x] GraphRAG chạy
- [x] Super-node check
- [x] Golden Dataset có gold answers thật
- [x] Evaluation chạy hết
- [x] Export results + summary CSV
- [x] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau